In [1]:
def retrieve_top_k(
    query,
    collection,
    embed_model,
    top_k=5,
    where=None,
):
    query = query.strip()

    if not query:
        raise ValueError("검색 질문이 비어 있습니다.")

    if top_k < 1:
        raise ValueError("top_k는 1 이상이어야 합니다.")

    document_count = collection.count()

    if document_count == 0:
        return []

    # DB에 저장된 문서와 동일한 모델·설정으로 질문 임베딩
    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True,
    )

    if query_embedding.ndim != 2:
        raise ValueError(
            f"질문 임베딩 차원 오류: {query_embedding.shape}"
        )

    if query_embedding.shape[0] != 1:
        raise ValueError(
            f"질문은 1개여야 합니다: {query_embedding.shape}"
        )

    query_kwargs = {
        "query_embeddings": query_embedding.tolist(),
        "n_results": min(top_k, document_count),
        "include": [
            "documents",
            "metadatas",
            "distances",
        ],
    }

    if where:
        query_kwargs["where"] = where

    raw_results = collection.query(**query_kwargs)

    results = []

    for index, chunk_id in enumerate(raw_results["ids"][0]):
        distance = raw_results["distances"][0][index]

        results.append(
            {
                "rank": index + 1,
                "id": chunk_id,
                "page_content": raw_results["documents"][0][index],
                "metadata": raw_results["metadatas"][0][index],
                "distance": distance,
                "score": 1.0 - distance,
            }
        )

    return results

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer


EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"
CHROMA_PATH = "../data/chroma_db"
COLLECTION_NAME = "maple_probability_row_v1"


embed_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

collection = client.get_collection(
    name=COLLECTION_NAME
)

c:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2992.13it/s]


NotFoundError: Collection [maple_probability_row_v1] does not exist

In [ ]:
results = retrieve_top_k(
    query="스타 미니콘의 획득 확률은?",
    collection=collection,
    embed_model=embed_model,
    top_k=5,
)

for result in results:
    print("=" * 80)
    print(f"순위: {result['rank']}")
    print(f"점수: {result['score']:.4f}")
    print(f"출처: {result['metadata'].get('url')}")
    print(f"문서명: {result['metadata'].get('name')}")
    print()
    print(result["page_content"])